# Timed Directional Network Walkthrough

This notebook runs the complete Virar-Dadar simulation with 30-second ticks, SLOW/FAST/EXPRESS services, scheduled stops, persistent block occupancy, directional loops, and priority overtaking. Run it from top to bottom after changing core files.

In [6]:
import subprocess
import sys
from pathlib import Path

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'scenarios').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root.')

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from models.station import Station
from models.train import EXPRESS, FAST, SLOW, TRAIN_PROFILES, Train
from scenarios.scenario_1 import build_trains
from src.planning.actions import ActionType
from src.planning.scheduler import Scheduler
from src.railway.network import DOWN, UP, Network, default_network
from src.railway.occupancy import OccupancyState
from src.simulation.simulator import TICK_SECONDS, Simulator

repo_root


WindowsPath('C:/Users/mikhi/OneDrive/Desktop/Incomplete Projects/Train_Traffic_Control')

In [7]:
def train_table(simulator):
    metrics = simulator.get_metrics()['trains']
    return [
        {
            'name': train.name,
            'type': train.train_type,
            'status': metrics[train.name]['status'],
            'station': simulator.network.station_name(train.current_station),
            'target': metrics[train.name]['target_station'],
            'line': train.line,
            'travel_ticks_left': train.remaining_travel_ticks,
            'dwell_ticks_left': train.dwell_remaining_ticks,
            'waiting_seconds': metrics[train.name]['waiting_seconds'],
            'loop_entries': train.loop_entries,
        }
        for train in simulator.trains
    ]

def action_table(actions):
    return [
        {
            'train': action.train_name,
            'action': action.action_type.value,
            'source_line': action.source_track,
            'target_line': action.target_track,
            'block': action.block,
            'reason': action.reason,
            'conflict': action.conflict,
        }
        for action in actions
    ]


## 1. Timing and service profiles

SLOW and FAST have equal block running times. FAST gains time only by skipping configured station dwells. EXPRESS traverses blocks faster and also has a smaller stop pattern.

In [8]:
[
    {
        'type': train_type,
        'tick_seconds': TICK_SECONDS,
        'block_running_ticks': profile.running_ticks,
        'block_running_seconds': profile.running_ticks * TICK_SECONDS,
        'scheduled_dwell_seconds': profile.dwell_ticks * TICK_SECONDS,
    }
    for train_type, profile in TRAIN_PROFILES.items()
]


[{'type': 'SLOW',
  'tick_seconds': 30,
  'block_running_ticks': 2,
  'block_running_seconds': 60,
  'scheduled_dwell_seconds': 30},
 {'type': 'FAST',
  'tick_seconds': 30,
  'block_running_ticks': 2,
  'block_running_seconds': 60,
  'scheduled_dwell_seconds': 30},
 {'type': 'EXPRESS',
  'tick_seconds': 30,
  'block_running_ticks': 1,
  'block_running_seconds': 30,
  'scheduled_dwell_seconds': 30}]

## 2. Directional topology

Increasing station indexes are UP toward Dadar and Churchgate. Decreasing indexes are DOWN toward Virar. Borivali and Andheri have loops in both directions.

In [9]:
[
    {
        'index': index,
        'station': station.name,
        'up_main': True,
        'down_main': True,
        'up_loop': station.has_loop(UP),
        'down_loop': station.has_loop(DOWN),
    }
    for index, station in enumerate(default_network.stations)
]


[{'index': 0,
  'station': 'Virar',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False},
 {'index': 1,
  'station': 'Bhayandar',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False},
 {'index': 2,
  'station': 'Borivali',
  'up_main': True,
  'down_main': True,
  'up_loop': True,
  'down_loop': True},
 {'index': 3,
  'station': 'Andheri',
  'up_main': True,
  'down_main': True,
  'up_loop': True,
  'down_loop': True},
 {'index': 4,
  'station': 'Bandra',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False},
 {'index': 5,
  'station': 'Dadar',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False}]

In [10]:
up_block = default_network.block_between(2, 3)
down_block = default_network.block_between(3, 2)
block_state = OccupancyState(default_network)
block_state.reserve_block(up_block.key, 'UP_DEMO', 2, 3)
down_available, reason = block_state.can_reserve_block(
    down_block.key, 'DOWN_DEMO', 3, 2
)

{
    'up_block': up_block.key,
    'down_block': down_block.key,
    'parallel_keys_are_distinct': up_block.key != down_block.key,
    'down_available_while_up_reserved': down_available,
    'reason': reason,
}


{'up_block': (2, 3, 'up'),
 'down_block': (2, 3, 'down'),
 'parallel_keys_are_distinct': True,
 'down_available_while_up_reserved': True,
 'reason': 'directional block available'}

## 3. Default slow, fast, and express scenario

The UP express begins behind the UP slow. The DOWN fast runs independently and skips Bandra and Bhayandar.

In [11]:
trains = build_trains()
sim = Simulator(trains, Scheduler(), network=default_network, verbose=False)

{
    'tick_seconds': TICK_SECONDS,
    'trains': train_table(sim),
    'station_occupancy': sim.occupancy_state.snapshot()['stations'],
}


{'tick_seconds': 30,
 'trains': [{'name': 'UP_EXP1',
   'type': 'EXPRESS',
   'status': 'AT_STATION',
   'station': 'Virar',
   'target': None,
   'line': 'up_main',
   'travel_ticks_left': 0,
   'dwell_ticks_left': 0,
   'waiting_seconds': 0,
   'loop_entries': 0},
  {'name': 'UP_SLOW1',
   'type': 'SLOW',
   'status': 'AT_STATION',
   'station': 'Bhayandar',
   'target': None,
   'line': 'up_main',
   'travel_ticks_left': 0,
   'dwell_ticks_left': 0,
   'waiting_seconds': 0,
   'loop_entries': 0},
  {'name': 'DOWN_FAST1',
   'type': 'FAST',
   'status': 'AT_STATION',
   'station': 'Dadar',
   'target': None,
   'line': 'down_main',
   'travel_ticks_left': 0,
   'dwell_ticks_left': 0,
   'waiting_seconds': 0,
   'loop_entries': 0}],
 'station_occupancy': [{'station': 'Virar',
   'up_main': 'UP_EXP1',
   'down_main': None},
  {'station': 'Bhayandar', 'up_main': 'UP_SLOW1', 'down_main': None},
  {'station': 'Borivali',
   'up_main': None,
   'down_main': None,
   'up_loop': None,
   'do

In [12]:
tick_zero_actions = sim.step()
tick_zero_occupancy = sim.history[-1]['occupancy']

{
    'elapsed_seconds': sim.history[-1]['elapsed_seconds'],
    'actions': action_table(tick_zero_actions),
    'trains_after_tick': train_table(sim),
    'blocks_held_during_travel': tick_zero_occupancy['blocks'],
    'destination_lines_reserved': tick_zero_occupancy['line_reservations'],
}


{'elapsed_seconds': 30,
 'actions': [{'train': 'UP_EXP1',
   'action': 'WAIT',
   'source_line': 'up_main',
   'target_line': 'up_main',
   'block': None,
   'reason': 'target station up_main occupied or reserved by UP_SLOW1',
   'conflict': True},
  {'train': 'DOWN_FAST1',
   'action': 'MOVE',
   'source_line': 'down_main',
   'target_line': 'down_main',
   'block': (4, 5, 'down'),
   'reason': 'depart for 2-tick block traversal',
   'conflict': False},
  {'train': 'UP_SLOW1',
   'action': 'MOVE',
   'source_line': 'up_main',
   'target_line': 'up_main',
   'block': (1, 2, 'up'),
   'reason': 'depart for 2-tick block traversal',
   'conflict': False}],
 'trains_after_tick': [{'name': 'UP_EXP1',
   'type': 'EXPRESS',
   'status': 'AT_STATION',
   'station': 'Virar',
   'target': None,
   'line': 'up_main',
   'travel_ticks_left': 0,
   'dwell_ticks_left': 0,
   'waiting_seconds': 30,
   'loop_entries': 0},
  {'name': 'UP_SLOW1',
   'type': 'SLOW',
   'status': 'IN_TRANSIT',
   'station

In [13]:
assert sim.get_metrics()['total_time_seconds'] == 30
assert len(tick_zero_occupancy['blocks']) == 2
assert len(tick_zero_occupancy['line_reservations']) == 2
assert next(train for train in trains if train.name == 'UP_SLOW1').is_in_transit
assert next(train for train in trains if train.name == 'DOWN_FAST1').is_in_transit

'Local trains remain in their blocks after the first 30-second tick.'


'Local trains remain in their blocks after the first 30-second tick.'

## 4. Let the express catch the slow local

The express takes one tick per block while the slow takes two. At Borivali, the existing priority rule moves the slow into `up_loop`.

In [14]:
overtake_actions = []
while sim.active_trains() and not any(
    action.action_type == ActionType.ENTER_LOOP
    for action in overtake_actions
):
    overtake_actions = sim.step()

{
    'tick': sim.history[-1]['time'],
    'elapsed_seconds': sim.history[-1]['elapsed_seconds'],
    'actions': action_table(overtake_actions),
    'trains': train_table(sim),
}


{'tick': 2,
 'elapsed_seconds': 90,
 'actions': [{'train': 'UP_SLOW1',
   'action': 'ENTER_LOOP',
   'source_line': 'up_main',
   'target_line': 'up_loop',
   'block': None,
   'reason': 'enter up_loop at Borivali to allow UP_EXP1 to overtake',
   'conflict': False},
  {'train': 'UP_EXP1',
   'action': 'MOVE',
   'source_line': 'up_main',
   'target_line': 'up_main',
   'block': (1, 2, 'up'),
   'reason': 'depart for 1-tick block traversal',
   'conflict': False},
  {'train': 'DOWN_FAST1',
   'action': 'MOVE',
   'source_line': 'down_main',
   'target_line': 'down_main',
   'block': (3, 4, 'down'),
   'reason': 'depart for 2-tick block traversal',
   'conflict': False},
  {'train': 'UP_EXP1',
   'action': 'ARRIVE',
   'source_line': 'up_main',
   'target_line': 'up_main',
   'block': (1, 2, 'up'),
   'reason': 'arrived for scheduled 1-tick stop',
   'conflict': False}],
 'trains': [{'name': 'UP_EXP1',
   'type': 'EXPRESS',
   'status': 'DWELLING',
   'station': 'Borivali',
   'target':

In [15]:
decision_actions = [
    action for action in overtake_actions
    if action.action_type != ActionType.ARRIVE
]
by_train = {action.train_name: action for action in decision_actions}
assert by_train['UP_SLOW1'].action_type == ActionType.ENTER_LOOP
assert by_train['UP_EXP1'].action_type == ActionType.MOVE

'The faster express caught the slow local and triggered the safe loop overtake.'


'The faster express caught the slow local and triggered the safe loop overtake.'

In [16]:
while sim.active_trains():
    sim.step()

sim.get_metrics()


{'total_ticks': 13,
 'tick_seconds': 30,
 'total_time_seconds': 390,
 'arrived_trains': 3,
 'active_trains': 0,
 'throughput': 0.23076923076923078,
 'throughput_per_hour': 27.692307692307693,
 'conflict_count': 2,
 'loop_usage': 1,
 'trains': {'UP_EXP1': {'finished': True,
   'waiting_time': 1,
   'waiting_seconds': 30,
   'completion_time': 7,
   'completion_time_seconds': 210,
   'current_station': 'Dadar',
   'target_station': None,
   'line': 'up_main',
   'train_type': 'EXPRESS',
   'status': 'FINISHED',
   'remaining_travel_ticks': 0,
   'dwell_remaining_ticks': 0,
   'loop_entries': 0},
  'UP_SLOW1': {'finished': True,
   'waiting_time': 1,
   'waiting_seconds': 30,
   'completion_time': 13,
   'completion_time_seconds': 390,
   'current_station': 'Dadar',
   'target_station': None,
   'line': 'up_main',
   'train_type': 'SLOW',
   'status': 'FINISHED',
   'remaining_travel_ticks': 0,
   'dwell_remaining_ticks': 0,
   'loop_entries': 1},
  'DOWN_FAST1': {'finished': True,
   'wa

In [17]:
timeline = [
    {
        'tick': tick['time'],
        'elapsed_seconds': tick['elapsed_seconds'],
        **row,
    }
    for tick in sim.history
    for row in action_table(tick['actions'])
]

timeline


[{'tick': 0,
  'elapsed_seconds': 30,
  'train': 'UP_EXP1',
  'action': 'WAIT',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': None,
  'reason': 'target station up_main occupied or reserved by UP_SLOW1',
  'conflict': True},
 {'tick': 0,
  'elapsed_seconds': 30,
  'train': 'DOWN_FAST1',
  'action': 'MOVE',
  'source_line': 'down_main',
  'target_line': 'down_main',
  'block': (4, 5, 'down'),
  'reason': 'depart for 2-tick block traversal',
  'conflict': False},
 {'tick': 0,
  'elapsed_seconds': 30,
  'train': 'UP_SLOW1',
  'action': 'MOVE',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': (1, 2, 'up'),
  'reason': 'depart for 2-tick block traversal',
  'conflict': False},
 {'tick': 1,
  'elapsed_seconds': 60,
  'train': 'UP_EXP1',
  'action': 'MOVE',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': (0, 1, 'up'),
  'reason': 'depart for 1-tick block traversal',
  'conflict': False},
 {'tick': 1,
  'elapsed_seconds': 60,
  'train': 

In [18]:
[
    row
    for row in timeline
    if row['action'] in {ActionType.ARRIVE.value, ActionType.DWELL.value}
]


[{'tick': 1,
  'elapsed_seconds': 60,
  'train': 'UP_EXP1',
  'action': 'ARRIVE',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': (0, 1, 'up'),
  'reason': 'passed station without a scheduled dwell',
  'conflict': False},
 {'tick': 1,
  'elapsed_seconds': 60,
  'train': 'UP_SLOW1',
  'action': 'ARRIVE',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': (1, 2, 'up'),
  'reason': 'arrived for scheduled 1-tick stop',
  'conflict': False},
 {'tick': 1,
  'elapsed_seconds': 60,
  'train': 'DOWN_FAST1',
  'action': 'ARRIVE',
  'source_line': 'down_main',
  'target_line': 'down_main',
  'block': (4, 5, 'down'),
  'reason': 'passed station without a scheduled dwell',
  'conflict': False},
 {'tick': 2,
  'elapsed_seconds': 90,
  'train': 'UP_EXP1',
  'action': 'ARRIVE',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': (1, 2, 'up'),
  'reason': 'arrived for scheduled 1-tick stop',
  'conflict': False},
 {'tick': 3,
  'elapsed_seconds': 120,
 

## 5. Editable experiment

Change train types, starting positions, priorities, or `scheduled_stops`. SLOW stops everywhere; FAST and EXPRESS stop only at their configured indexes plus their destination.

In [19]:
experiment_trains = [
    Train('MY_EXPRESS', EXPRESS, 10, 0, 5, scheduled_stops=(2, 5)),
    Train('MY_SLOW', SLOW, 1, 1, 5),
    Train('MY_FAST', FAST, 5, 5, 0, scheduled_stops=(3, 2, 0)),
]
experiment = Simulator(
    experiment_trains, Scheduler(), network=default_network, verbose=False
)
experiment_metrics = experiment.run(max_ticks=50)

{
    'metrics': experiment_metrics,
    'timeline': [
        {'tick': tick['time'], 'elapsed_seconds': tick['elapsed_seconds'], **row}
        for tick in experiment.history
        for row in action_table(tick['actions'])
    ],
}


{'metrics': {'total_ticks': 13,
  'tick_seconds': 30,
  'total_time_seconds': 390,
  'arrived_trains': 3,
  'active_trains': 0,
  'throughput': 0.23076923076923078,
  'throughput_per_hour': 27.692307692307693,
  'conflict_count': 2,
  'loop_usage': 1,
  'trains': {'MY_EXPRESS': {'finished': True,
    'waiting_time': 1,
    'waiting_seconds': 30,
    'completion_time': 7,
    'completion_time_seconds': 210,
    'current_station': 'Dadar',
    'target_station': None,
    'line': 'up_main',
    'train_type': 'EXPRESS',
    'status': 'FINISHED',
    'remaining_travel_ticks': 0,
    'dwell_remaining_ticks': 0,
    'loop_entries': 0},
   'MY_SLOW': {'finished': True,
    'waiting_time': 1,
    'waiting_seconds': 30,
    'completion_time': 13,
    'completion_time_seconds': 390,
    'current_station': 'Dadar',
    'target_station': None,
    'line': 'up_main',
    'train_type': 'SLOW',
    'status': 'FINISHED',
    'remaining_travel_ticks': 0,
    'dwell_remaining_ticks': 0,
    'loop_entries

## 6. Run the complete test suite

Run this after editing core files. The cell fails if any timing, occupancy, direction, dwell, or overtaking test breaks.

In [20]:
test_run = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-v'],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(test_run.stdout)
print(test_run.stderr)
assert test_run.returncode == 0, 'Core tests failed.'
'All core tests passed.'



test_arrival_line_reservation_blocks_loop_exit (tests.test_core_engine.CoreEngineTest.test_arrival_line_reservation_blocks_loop_exit) ... ok
test_default_mixed_direction_scenario_completes_safely (tests.test_core_engine.CoreEngineTest.test_default_mixed_direction_scenario_completes_safely) ... ok
test_default_network_maps_virar_to_dadar_as_up (tests.test_core_engine.CoreEngineTest.test_default_network_maps_virar_to_dadar_as_up) ... ok
test_directional_loops_exist_only_where_configured (tests.test_core_engine.CoreEngineTest.test_directional_loops_exist_only_where_configured) ... ok
test_express_crosses_one_block_in_one_tick (tests.test_core_engine.CoreEngineTest.test_express_crosses_one_block_in_one_tick) ... ok
test_higher_priority_down_train_overtakes_through_down_loop (tests.test_core_engine.CoreEngineTest.test_higher_priority_down_train_overtakes_through_down_loop) ... ok
test_higher_priority_up_train_overtakes_through_up_loop (tests.test_core_engine.CoreEngineTest.test_higher_prio

'All core tests passed.'